In [ ]:
"""
Inventory Order Recommendation Engine

This notebook demonstrates:
- Sales velocity calculation
- ABC classification using Pareto distribution
- Rule-based ordering logic

Data sources: /data folder
"""

In [1]:
# Import packages and functions.

import numpy as np
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path

In [2]:
# Read csv files.

# Use raw strings (r"...") to prevent Python from interpreting backslashes '\' as escape sequences (e.g., \n, \t, \uXXXX).
filepath = Path(r"C:\Users\jerre\Dropbox\Public\Portfolio\Gift Shop Retail\Retail Store 2 - Excel Files")

# Use encoding="latin1" because the source CSV files are not UTF-8 encoded, which is common with Excel/Windows exports.
# product_id is a key used to join tables and must remain a string to preserve integrity.
read_args = {"encoding": "latin1",
            "dtype": {"product_id": str}}

def load_csv(name):
    return pd.read_csv(filepath / f"{name}.csv", **read_args)

tblProduct = load_csv("tblProduct")
tblSales = load_csv("tblSales")
tblPurchases = load_csv("tblPurchases")
tblInventorySnapshot = load_csv("tblInventorySnapshot")

In [57]:
# Display formatting.

def style_report_table(df, left_align_cols=None):

    # Normalize inputs
    if left_align_cols is None:
        left_align_cols = []

    # Build base styler
    styler = df.style

    # Apply column formatting
    format_dict = {
        "daily_veloc": "{:.2f}",
        "weekly_veloc": "{:.2f}",
        "pct_of_sales": "{:.2%}",
        "cum_pct_of_sales": "{:.2%}",
        "cur_dep_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "proj_dep_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "depletion_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "cost": "${:,.2f}",
        "avg_unit_cost": "${:,.2f}",
        "retail_price": "${:,.2f}",
        "received_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "sale_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "snapshot_date": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "period_start": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else "",
        "period_end": lambda x: x.strftime("%Y-%m-%d") if pd.notnull(x) else ""        
    }

    existing_formats = {col: fmt for col, fmt in format_dict.items() if col in df.columns}

    if existing_formats:
        styler = styler.format(existing_formats)

    # Left-align selected columns
    # and their headers
    existing_left_align_cols = [col for col in left_align_cols if col in df.columns]

    if existing_left_align_cols:
        styler = styler.set_properties(
            subset=existing_left_align_cols,
            **{"text-align": "left"}
        )

        header_styles = []
        for col in existing_left_align_cols:
            col_idx = df.columns.get_loc(col)
            header_styles.append({
                "selector": f"th.col_heading.level0.col{col_idx}",
                "props": [("text-align", "left")]
            })

        styler = styler.set_table_styles(header_styles, overwrite=False)

    # Enable Header Wrapping
    styler = styler.set_table_styles(
    [{
        "selector": "th",
        "props": [("white-space", "pre-line")]
    }],
    overwrite=False
    )

    # Highlight ABC rows
    if "abc_class" in df.columns or "abc_desc" in df.columns:

        def highlight_abc_rows(row):
            # prefer abc_class if available
            val = ""
            if "abc_class" in row.index:
                val = str(row["abc_class"])
            elif "abc_desc" in row.index:
                val = str(row["abc_desc"])

            if val.startswith("A"):
                return ["background-color: #D4EDDA"] * len(row)
            elif val.startswith("B"):
                return ["background-color: #FFF3CD"] * len(row)
            elif val.startswith("C"):
                return ["background-color: #F8D7DA"] * len(row)

            return [""] * len(row)

        styler = styler.apply(highlight_abc_rows, axis=1)

    return styler

In [4]:
# Reporting period parameters and derived metrics. 

# User defined parameters for the reporting period.
period_start = pd.to_datetime("2025-12-31")
period_end = pd.to_datetime("2026-02-03")
delivery_date = pd.to_datetime("2026-02-09")
next_delivery_date = pd.to_datetime("2026-02-23")

# Derived period and delivery metrics.
period_days = (period_end - period_start).days + 1
lead_time = (delivery_date - period_end).days
days_to_next_delivery = (next_delivery_date - period_end).days
days_between_deliveries = (next_delivery_date - delivery_date).days

# Flag for minimum length of reporting period. Shorter periods result in data and metrics that are unreliable for ordering calculations.  
if period_days >= 14:
    planning_status = "Valid planning window (Period Days >= 14 days)"
else:
    planning_status = "Window too short for planning (Period Days < 14 days)"

# Validation output.
print(f"Period Days = {period_days}")
print(f"Lead Time = {lead_time}")
print(f"Days to next delivery = {days_to_next_delivery}")
print(f"Days between deliveries = {days_between_deliveries}")
print(f"Note: {planning_status}")

Period Days = 35
Lead Time = 6
Days to next delivery = 20
Days between deliveries = 14
Note: Valid planning window (Period Days >= 14 days)


In [5]:
# Build the base table by combining the product master
# with beginning inventory, purchases, and sales for the reporting period.

In [6]:


# Ensure snapshot_date is in datetime format for accurate filtering.
tblInventorySnapshot["snapshot_date"] = pd.to_datetime(tblInventorySnapshot["snapshot_date"])

# SELECT product_id, qty_on_hand AS beg_qty
# FROM tblInventorySnapshot
# WHERE snapshot_date = period_start

mask = tblInventorySnapshot["snapshot_date"] == period_start

beg_inventory = (tblInventorySnapshot.loc[mask, ["product_id", "qty_on_hand"]]
    .rename(columns={"qty_on_hand": "beg_qty"})
)

# Validation output.
beg_inventory.head()

,product_id,beg_qty
0,1088638,30
1,1253167,12
2,1331817,30
3,1711217,26
4,2012714,14


In [7]:
# Ensure received_date is in datetime format for accurate filtering.
tblPurchases["received_date"] = pd.to_datetime(tblPurchases["received_date"])

# SELECT product_id, SUM(total_units) AS purchased
# FROM tblPurchases
# WHERE received_date BETWEEN period_start AND period_end
# GROUP BY product_id

mask = tblPurchases["received_date"].between(period_start, period_end)

purchases = (tblPurchases.loc[mask, ["product_id", "total_qty_purchased"]]
    .groupby("product_id", as_index=False)["total_qty_purchased"].sum()
    .rename(columns={"total_qty_purchased": "purchased"})
)



# Validation output.
purchases.head()

,product_id,purchased
0,1088638,72
1,1253167,96
2,1711217,96
3,2732499,72
4,3894456,24


In [8]:
# Ensure sale_date is in datetime format for accurate filtering.
tblSales["sale_date"] = pd.to_datetime(tblSales["sale_date"])

# SELECT product_id, SUM(quantity) AS sold
# FROM tblSales
# WHERE sale_date BETWEEN period_start AND period_end
# GROUP BY product_id

mask = tblSales["sale_date"].between(period_start, period_end)

sales = (tblSales.loc[mask, ["product_id", "quantity"]]    
    .groupby("product_id", as_index=False)["quantity"].sum()
    .rename(columns={"quantity": "sold"})  
)

# Validation output.
sales.head()

,product_id,sold
0,1088638,79
1,1253167,62
2,1331817,24
3,1711217,87
4,2012714,10


In [9]:
# Build the base table by left joining the product master
# to beginning inventory, purchases, and sales.

# SELECT 
#   p.product_id, 
#   p.product_name, 
#   p.units_per_order_unit,
#   COALESCE(b.beg_qty, 0) AS beg_qty,
#   COALESCE(pr.purchased, 0) AS purchased,
#   COALESCE(s.sold, 0) AS sold
# FROM tblProduct p
# LEFT JOIN beg_inventory b 
#   ON p.product_id = b.product_id
# LEFT JOIN purchases pr
#   ON p.product_id = pr.product_id
# LEFT JOIN sales s
#   ON p.product_id = s.product_id
base = (
    tblProduct[["product_id", "product_name", "purchase_unit_size"]]
    .merge(beg_inventory, on="product_id", how="left")
    .merge(purchases, on="product_id", how="left")
    .merge(sales, on="product_id", how="left")
    .fillna({
        "beg_qty": 0,
        "purchased": 0,
        "sold": 0
    })
)

# Left joins introduce missing values for products with no inventory, purchases, or sales.
# Pandas stores missing numeric values as NaN, which causes these columns to become floats.
# After filling missing values with 0, cast the columns back to integers.
base[["beg_qty", "purchased", "sold"]] = (
    base[["beg_qty", "purchased", "sold"]].astype(int)
)

base = base.sort_values(
    by=["sold", "product_id"],
    ascending=[False, True]
).reset_index(drop=True)

# Validation output.
display_columns = [
    "product_id",
    "product_name",
    "purchase_unit_size",
    "beg_qty",
    "purchased",
    "sold"
]
display(
    style_report_table(
        base[display_columns].head(),
        left_align_cols=["product_name", "abc_desc"]
    )
)

,product_id,product_name,purchase_unit_size,beg_qty,purchased,sold
0,9292315,Sour Patch Kids,28,28,112,90
1,1711217,Ruffles Cheddar & Sour Cream,24,26,96,87
2,1088638,Protein Bar,24,30,72,79
3,2732499,Ritz Crackers Snack Pack,24,16,72,68
4,1253167,Lay's Classic Potato Chips,24,12,96,62


In [10]:
# Extend base to create sales_metrics with inventory, velocity, depletion, and sales-share metrics.
sales_metrics = base.copy()

sales_metrics["end_qty"] = (
    sales_metrics["beg_qty"]
    + sales_metrics["purchased"]
    - sales_metrics["sold"]
)

sales_metrics["daily_veloc"] = sales_metrics["sold"] / period_days

sales_metrics["weekly_veloc"] = sales_metrics["daily_veloc"] * 7

# Calculate current days of inventory.
sales_metrics["cur_days_of_inv"] = (
    sales_metrics["end_qty"] 
    / sales_metrics["daily_veloc"].replace(0, np.nan)
)

# Floor current days of inventory to whole days.
# Cast to pandas nullable Int64 for integer representation while retaining missing values (<NA>).
# Note: NumPy int64 (lowercase 'i') cannot be used because it does not support null values.
sales_metrics["cur_days_of_inv"] = (
    np.floor(sales_metrics["cur_days_of_inv"])
).astype("Int64")

# Project the current depletion date.
sales_metrics["cur_dep_date"] = (
    period_end 
    + pd.to_timedelta(sales_metrics["cur_days_of_inv"], unit="D")
)

# Validation output.
display_columns = [
    "product_name",
    "beg_qty",
    "purchased",
    "sold",
    "end_qty",
    "daily_veloc",
    "weekly_veloc",
    "cur_days_of_inv",
    "cur_dep_date"
]

display(
    style_report_table(
        sales_metrics[display_columns].head(),
        left_align_cols=["product_name"]
    )
)

,product_name,beg_qty,purchased,sold,end_qty,daily_veloc,weekly_veloc,cur_days_of_inv,cur_dep_date
0,Sour Patch Kids,28,112,90,50,2.57,18.00,19,2026-02-22
1,Ruffles Cheddar & Sour Cream,26,96,87,35,2.49,17.40,14,2026-02-17
2,Protein Bar,30,72,79,23,2.26,15.80,10,2026-02-13
3,Ritz Crackers Snack Pack,16,72,68,20,1.94,13.60,10,2026-02-13
4,Lay's Classic Potato Chips,12,96,62,46,1.77,12.40,25,2026-02-28


In [11]:
# Extend sales_metrics to create abc_classification using Pareto-based cumulative percent of sales.

abc_classification = sales_metrics.copy()

# Before calculating cumulative percent of sales, sort by sold descending.
# product_id is used as a secondary sort key to break ties consistently.
abc_classification = abc_classification.sort_values(
    by=["sold", "product_id"],
    ascending=[False, True]
).reset_index(drop=True)

# Calculate each product's share of total units sold in the reporting period.
total_sold = abc_classification["sold"].sum()
abc_classification["pct_of_sales"] = abc_classification["sold"] / total_sold

abc_classification["cum_pct_of_sales"] = abc_classification["pct_of_sales"].cumsum()

# Pareto classification based on cumulative percent of sales.
# Products contributing to the first 80% of cumulative sales are Class A.
# Products contributing to the next 15% are Class B.
# The remaining products are Class C.
abc_classification["abc_class"] = np.select(
    [
        abc_classification["cum_pct_of_sales"] <= 0.80,
        abc_classification["cum_pct_of_sales"] <= 0.95
    ],
    ["A", "B"],
    default="C"
)

# Inventory policy by ABC class:
# A: +14 days, B: +7 days, C: no additional inventory
classification_dictionary = {
    "A": {
        "desc": "A – Top Sellers: 2 weeks of extra inventory.",
        "targ_days": days_between_deliveries + 14
    },
    "B": {
        "desc": "B – Mid Sellers: 1 week of extra inventory.",
        "targ_days": days_between_deliveries + 7
    },
    "C": {
        "desc": "C – Low Sellers: no extra inventory.",
        "targ_days": days_between_deliveries
    }
}

abc_classification["abc_desc"] = (
    abc_classification["abc_class"].map(lambda x: classification_dictionary[x]["desc"])
)

abc_classification["targ_days_of_inv_after_delivery"] = (
    abc_classification["abc_class"].map(lambda x: classification_dictionary[x]["targ_days"])
)

# Validation output.
display_columns = [
    "product_name",
    "sold",
    "pct_of_sales",
    "cum_pct_of_sales",
    "abc_desc",
    "targ_days_of_inv_after_delivery"
]
display(
    style_report_table(
        abc_classification[display_columns],
        left_align_cols=["product_name", "abc_desc"]
    )
)

,product_name,sold,pct_of_sales,cum_pct_of_sales,abc_desc,targ_days_of_inv_after_delivery
0,Sour Patch Kids,90,10.99%,10.99%,A – Top Sellers: 2 weeks of extra inventory.,28
1,Ruffles Cheddar & Sour Cream,87,10.62%,21.61%,A – Top Sellers: 2 weeks of extra inventory.,28
2,Protein Bar,79,9.65%,31.26%,A – Top Sellers: 2 weeks of extra inventory.,28
3,Ritz Crackers Snack Pack,68,8.30%,39.56%,A – Top Sellers: 2 weeks of extra inventory.,28
4,Lay's Classic Potato Chips,62,7.57%,47.13%,A – Top Sellers: 2 weeks of extra inventory.,28
5,Trail Mix,52,6.35%,53.48%,A – Top Sellers: 2 weeks of extra inventory.,28
6,Nerds,49,5.98%,59.46%,A – Top Sellers: 2 weeks of extra inventory.,28
7,Snickers,48,5.86%,65.32%,A – Top Sellers: 2 weeks of extra inventory.,28
8,Skittles,38,4.64%,69.96%,A – Top Sellers: 2 weeks of extra inventory.,28
9,Pringles Original,34,4.15%,74.11%,A – Top Sellers: 2 weeks of extra inventory.,28


In [12]:
# Extend abc_classification to create order_recommendations with delivery, target inventory,
# units-to-order, and purchase-unit recommendations.

order_recommendations = abc_classification.copy()

# Project inventory remaining at the delivery date.
order_recommendations["inv_at_deliv"] = np.maximum(
    order_recommendations["end_qty"]
    - np.ceil(order_recommendations["daily_veloc"] * lead_time),
    0
).astype(int)

# Calculate the target inventory level immediately after delivery.
order_recommendations["targ_inv_after_deliv"] = np.ceil(
    order_recommendations["targ_days_of_inv_after_delivery"]
    * order_recommendations["daily_veloc"]
).astype(int)

# Units needed to reach the target inventory level after delivery.
order_recommendations["units_to_order"] = np.maximum(
    order_recommendations["targ_inv_after_deliv"]
    - order_recommendations["inv_at_deliv"],
    0
).astype(int)

# Convert required units into purchasable order units (cases, packs, etc.),
# rounding up and returning null when units_per_order_unit is null or zero.
order_recommendations["purch_units_to_order"] = (
    np.ceil(
        order_recommendations["units_to_order"]
        / order_recommendations["purchase_unit_size"].replace(0, np.nan)
    ).astype("Int64")
)

# Validation output.
display_columns = [
    "product_name",
    "abc_class",
    "inv_at_deliv",
    "targ_inv_after_deliv",
    "units_to_order",
    "purchase_unit_size",
    "purch_units_to_order"
]
display(
    style_report_table(
        order_recommendations[display_columns],
        left_align_cols=["product_name", "abc_desc"]
    )
)

,product_name,abc_class,inv_at_deliv,targ_inv_after_deliv,units_to_order,purchase_unit_size,purch_units_to_order
0,Sour Patch Kids,A,34,72,38,28,2
1,Ruffles Cheddar & Sour Cream,A,20,70,50,24,3
2,Protein Bar,A,9,64,55,24,3
3,Ritz Crackers Snack Pack,A,8,55,47,24,2
4,Lay's Classic Potato Chips,A,35,50,15,24,1
5,Trail Mix,A,2,42,40,24,2
6,Nerds,A,69,40,0,28,0
7,Snickers,A,12,39,27,28,1
8,Skittles,A,18,31,13,28,1
9,Pringles Original,A,21,28,7,24,1


In [13]:
# Extend order_recommendations to project post-delivery inventory,
# the number of days until depletion after delivery,
# and the projected depletion date.
updated_inventory_projection = order_recommendations.copy()

updated_inventory_projection["proj_qty_on_hand_after_deliv"] = (
    updated_inventory_projection["inv_at_deliv"]
    + (
        updated_inventory_projection["purch_units_to_order"] 
        * updated_inventory_projection["purchase_unit_size"]
    )
)

updated_inventory_projection["proj_days_until_dep_after_deliv"] = (
    np.floor(
        updated_inventory_projection["proj_qty_on_hand_after_deliv"]
        / updated_inventory_projection["daily_veloc"].replace(0, np.nan)
    )
).astype("Int64")

# Project the depletion date after delivery.
updated_inventory_projection["proj_dep_date"] = (
    delivery_date 
    + pd.to_timedelta(updated_inventory_projection["proj_days_until_dep_after_deliv"], unit="D")
)

# If nothing is being ordered, 
# then the projected depletion date should remain the current depletion date. 
updated_inventory_projection.loc[
    updated_inventory_projection["purch_units_to_order"] == 0,
    "proj_dep_date"
] = updated_inventory_projection["cur_dep_date"]


# Validation output.
display_columns = [
    "product_name",
    "inv_at_deliv",
    "targ_inv_after_deliv",
    "units_to_order",
    "purchase_unit_size",
    "purch_units_to_order",
    "proj_qty_on_hand_after_deliv",
    "proj_days_until_dep_after_deliv",
    "proj_dep_date"    
]
display(
    style_report_table(
        updated_inventory_projection[display_columns],
        left_align_cols=["product_name", "abc_desc"]
    )
)

,product_name,inv_at_deliv,targ_inv_after_deliv,units_to_order,purchase_unit_size,purch_units_to_order,proj_qty_on_hand_after_deliv,proj_days_until_dep_after_deliv,proj_dep_date
0,Sour Patch Kids,34,72,38,28,2,90,35,2026-03-16
1,Ruffles Cheddar & Sour Cream,20,70,50,24,3,92,37,2026-03-18
2,Protein Bar,9,64,55,24,3,81,35,2026-03-16
3,Ritz Crackers Snack Pack,8,55,47,24,2,56,28,2026-03-09
4,Lay's Classic Potato Chips,35,50,15,24,1,59,33,2026-03-14
5,Trail Mix,2,42,40,24,2,50,33,2026-03-14
6,Nerds,69,40,0,28,0,69,49,2026-03-30
7,Snickers,12,39,27,28,1,40,29,2026-03-10
8,Skittles,18,31,13,28,1,46,42,2026-03-23
9,Pringles Original,21,28,7,24,1,45,46,2026-03-27


In [14]:
# Display only the orders that need to be place. 
mask = updated_inventory_projection["purch_units_to_order"] != 0
orders_to_place = updated_inventory_projection.loc[mask]

# Columns to display (internal schema)
display_columns = [
    "product_id", 
    "product_name", 
    "end_qty", 
    "daily_veloc",
    "cur_dep_date",
    "abc_class",
    "purch_units_to_order",
    "proj_qty_on_hand_after_deliv",
    "proj_dep_date"  
]

# Build styled table using original column names (for formatting logic)
styled = style_report_table(
    orders_to_place[display_columns],
    left_align_cols=["product_name"]
)

# Mapping from internal names → business-friendly labels
rename_map = {
    "product_id": "Product ID",
    "product_name": "Product",
    "end_qty": "On Hand",
    "daily_veloc": "Avg Daily Sales",
    "cur_dep_date": "Depletion Date",
    "abc_class": "ABC Class",
    "purch_units_to_order": "Cases to Order",
    "proj_qty_on_hand_after_deliv": "Projected On Hand",
    "proj_dep_date": "Projected Depletion"
}

# Apply column renaming at the Styler level
display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in orders_to_place[display_columns].columns],
        axis=1
    )
)

,Product ID,Product,On Hand,Avg Daily Sales,Depletion Date,ABC Class,Cases to Order,Projected On Hand,Projected Depletion
0,9292315,Sour Patch Kids,50,2.57,2026-02-22,A,2,90,2026-03-16
1,1711217,Ruffles Cheddar & Sour Cream,35,2.49,2026-02-17,A,3,92,2026-03-18
2,1088638,Protein Bar,23,2.26,2026-02-13,A,3,81,2026-03-16
3,2732499,Ritz Crackers Snack Pack,20,1.94,2026-02-13,A,2,56,2026-03-09
4,1253167,Lay's Classic Potato Chips,46,1.77,2026-02-28,A,1,59,2026-03-14
5,7023463,Trail Mix,11,1.49,2026-02-10,A,2,50,2026-03-14
7,7165581,Snickers,21,1.37,2026-02-18,A,1,40,2026-03-10
8,6334728,Skittles,25,1.09,2026-02-26,A,1,46,2026-03-23
9,6825224,Pringles Original,27,0.97,2026-03-02,A,1,45,2026-03-27
10,8306905,Haribo Goldbears,18,0.89,2026-02-23,A,1,40,2026-03-26


In [15]:
# Display Class C products that may need review.
mask = updated_inventory_projection["abc_class"] == "C"
products_to_evaluate = updated_inventory_projection.loc[mask]

display_columns = [
    "product_id", 
    "product_name", 
    "abc_class",
    "end_qty", 
    "daily_veloc",
    "weekly_veloc",
    "proj_dep_date"
]
display(
    style_report_table(
        products_to_evaluate[display_columns],
        left_align_cols=["product_name", "abc_desc"]        
    )
)

,product_id,product_name,abc_class,end_qty,daily_veloc,weekly_veloc,proj_dep_date
19,8551457,Snyder's Pretzels,C,12,0.26,1.80,2026-03-21
20,2858379,Kit Kat,C,7,0.20,1.40,2026-03-10
21,2424707,Jelly Belly Jelly Beans,C,22,0.17,1.20,2026-06-11
22,7944400,Tootsie Roll,C,16,0.17,1.20,2026-05-07
23,9044404,Granola Bar,C,14,0.14,1.00,2026-05-12
24,8860153,Mike and Ike,C,3,0.11,0.80,2026-03-01
25,2982488,Popcorn (Microwave Bag),C,14,0.09,0.60,2026-07-16
26,6341525,Goldfish Crackers,C,5,0.06,0.40,2026-05-01
27,9115955,Laffy Taffy,C,13,0.06,0.40,2026-09-18
28,6230104,Hershey's Milk Chocolate Bar,C,22,0.03,0.20,2028-03-14


In [53]:
# Display only the orders that need to be place. 
# mask = updated_inventory_projection["purch_units_to_order"] != 0
# orders_to_place = updated_inventory_projection.loc[mask]

porfolio_abc_sample = (
    updated_inventory_projection
    .groupby("abc_class", group_keys=False)
    .head(3)
)

# Columns to display (internal schema)
display_columns = [
    "product_id", 
    "product_name", 
    "abc_class",
    "daily_veloc",
    "end_qty",     
    "cur_dep_date",
    "purch_units_to_order",
    "proj_qty_on_hand_after_deliv",
    "proj_dep_date"  
]

# Build styled table using original column names (for formatting logic)
styled = style_report_table(
    porfolio_abc_sample[display_columns],
    left_align_cols=["product_name"]
)

# Hide row indices. 
styled = styled.hide(axis="index")

# Verticle alighnment: bottom
styled = styled.set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [
                ("white-space", "pre-line"),
                ("vertical-align", "bottom")
            ]
        }
    ],
    overwrite=False
)

# Mapping from internal names → business-friendly labels
rename_map = {
    "product_id": " ProdID",
    "product_name": "Product",
    "abc_class": "ABC\nClass",
    "daily_veloc": "Avg\nDaily\nSales",
    "end_qty": "Currently\nOn Hand",
    "cur_dep_date": "Current\nDepletion\nDate",
    "purch_units_to_order": "Cases to\nOrder",
    "proj_qty_on_hand_after_deliv": "Projected\nOn Hand",
    "proj_dep_date": "Projected\nDepletion\nDate"
}

# Display user defined parameters
period_start_str = period_start.strftime('%Y-%m-%d')
period_end_str = period_end.strftime('%Y-%m-%d')
delivery_date_str = delivery_date.strftime('%Y-%m-%d')
next_delivery_date_str = next_delivery_date.strftime('%Y-%m-%d')

print()

display(HTML(f"""
<h2 style='margin-bottom: 10px;'>Products to Order</h2>

<table style='border-collapse: collapse; margin-bottom: 16px; font-size: 14px;'>
  <tr>
    <td style='padding: 6px 16px 6px 0; text-align: right;'><strong>Report Run Date:</strong></td>
    <td style='padding: 6px 28px 6px 0;'>{period_end_str}</td>

    <td style='padding: 6px 16px 6px 0; text-align: right;'><strong>Period Start:</strong></td>
    <td style='padding: 6px 28px 6px 0;'>{period_start_str}</td>

    <td style='padding: 6px 16px 6px 0; text-align: right;'><strong>Period End:</strong></td>
    <td style='padding: 6px 0;'>{period_end_str}</td>
  </tr>

  <tr style='background-color: white;'>
    <td></td>
    <td></td>

    <td style='padding: 6px 16px 6px 0; text-align: right;'><strong>Delivery Date:</strong></td>
    <td style='padding: 6px 28px 6px 0;'>{delivery_date_str}</td>

    <td style='padding: 6px 16px 6px 0; text-align: right;'><strong>Next Delivery Date:</strong></td>
    <td style='padding: 6px 0;'>{next_delivery_date_str}</td>
  </tr>
</table>
"""))

# Apply column renaming at the Styler level
display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in orders_to_place[display_columns].columns],
        axis=1
    )
)
print()

Report Run Date:,2026-02-03,Period Start:,2025-12-31,Period End:,2026-02-03
,,Delivery Date:,2026-02-09,Next Delivery Date:,2026-02-23


ProdID,Product,ABC Class,Avg Daily Sales,Currently On Hand,Current Depletion Date,Cases to Order,Projected On Hand,Projected Depletion Date
9292315,Sour Patch Kids,A,2.57,50,2026-02-22,2,90,2026-03-16
1711217,Ruffles Cheddar & Sour Cream,A,2.49,35,2026-02-17,3,92,2026-03-18
1088638,Protein Bar,A,2.26,23,2026-02-13,3,81,2026-03-16
3894456,Cheetos Crunchy,B,0.77,16,2026-02-23,1,35,2026-03-26
1331817,Planters Peanuts,B,0.69,6,2026-02-11,1,25,2026-03-17
7747619,Twix,B,0.60,14,2026-02-26,1,38,2026-04-13
8551457,Snyder's Pretzels,C,0.26,12,2026-03-21,0,10,2026-03-21
2858379,Kit Kat,C,0.20,7,2026-03-10,0,5,2026-03-10
2424707,Jelly Belly Jelly Beans,C,0.17,22,2026-06-11,0,20,2026-06-11


In [17]:
# Portfolio sample: tblProducts

# Columns to display
display_columns = [
    "product_id",
    "category",
    "product_name",
    "purchase_unit",
    "purchase_unit_size",
    "avg_unit_cost",
    "retail_price"
]

# Styled table
styled = style_report_table(
    tblProduct[display_columns].head(),
    left_align_cols=["product_id", "category", "product_name", "purchase_unit"]
)

# Hide row index
styled = styled.hide(axis="index")

# Verticle alighnment: bottom
styled = styled.set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [
                ("white-space", "pre-line"),
                ("vertical-align", "bottom")
            ]
        }
    ],
    overwrite=False
)

# Business-friendly column labels
rename_map = {
    "product_id": "Prod ID",
    "category": "Category",
    "product_name": "Product",
    "purchase_unit": "Unit\nType",
    "purchase_unit_size": "Units\nper Case",
    "avg_unit_cost": "Avg Unit\nCost",
    "retail_price": "Retail\nPrice"
}

# Display title
display(HTML("<h3 style='margin-bottom: 6px;'>Products Table</h3>"))

# Display renamed/styled table
display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in tblProduct[display_columns].columns],
        axis=1
    )
)

Prod ID,Category,Product,Unit Type,Units per Case,Avg Unit Cost,Retail Price
1088638,snack,Protein Bar,pack,24,$1.56,$3.49
1253167,snack,Lay's Classic Potato Chips,pack,24,$1.11,$2.69
1331817,snack,Planters Peanuts,pack,24,$0.96,$2.29
1711217,snack,Ruffles Cheddar & Sour Cream,pack,24,$1.16,$2.69
2012714,snack,Wheat Thins,pack,24,$1.11,$2.49


In [18]:
# Portfolio sample: tblPurchases

# Columns to display
display_columns = [
    "purchase_id",
    "product_id",
    "purchase_date",
    "received_date",
    "units_purchased",
    "total_qty_purchased",
    "cost"
]

# Styled table
styled = style_report_table(
    tblPurchases[display_columns].head(),
    left_align_cols=["purchase_id", "product_id"]
)

# Hide row index
styled = styled.hide(axis="index")

# Verticle alighnment: bottom
styled = styled.set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [
                ("white-space", "pre-line"),
                ("vertical-align", "bottom")
            ]
        }
    ],
    overwrite=False
)

# Business-friendly column labels
rename_map = {
    "purchase_id": "Purch ID",
    "product_id": "Prod ID",
    "purchase_date": "Purch\nDate",
    "received_date": "Received\nDate",
    "units_purchased": "Cases\nPurchased",
    "total_qty_purchased": "Units\nPurchased",
    "cost": "Cost"
}

# Display renamed/styled table
display(HTML("<h3>Purchases Table</h3>"))

display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in tblPurchases[display_columns].columns],
        axis=1
    )
)


Purch ID,Prod ID,Purch Date,Received Date,Cases Purchased,Units Purchased,Cost
2561424,1088638,1/2/2026,2026-01-05,3,72,$112.27
2561424,1253167,1/2/2026,2026-01-05,1,24,$26.56
2561424,1711217,1/2/2026,2026-01-05,2,48,$55.45
2561424,2732499,1/2/2026,2026-01-05,2,48,$50.71
2561424,4255235,1/2/2026,2026-01-05,1,24,$26.58


In [19]:
# Portfolio sample: tblSales

# Columns to display
display_columns = [
    "sale_id",
    "product_id",
    "sale_date",
    "quantity"
]

# Styled table
styled = style_report_table(
    tblSales[display_columns].head(),
    left_align_cols=["sale_id", "product_id"]
)

# Hide row index
styled = styled.hide(axis="index")

# Verticle alighnment: bottom
styled = styled.set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [
                ("white-space", "pre-line"),
                ("vertical-align", "bottom")
            ]
        }
    ],
    overwrite=False
)

# Business-friendly column labels
rename_map = {
    "sale_id": "Sale ID",
    "product_id": "Prod ID",
    "sale_date": "Sale\nDate",
    "quantity": "Units\nSold"
}

# Display title
display(HTML("<h3 style='margin-bottom: 6px;'>Sales Table</h3>"))

# Display renamed/styled table
display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in tblSales[display_columns].columns],
        axis=1
    )
)

Sale ID,Prod ID,Sale Date,Units Sold
100003,1088638,2025-12-31,1
100009,1088638,2025-12-31,2
100004,1253167,2025-12-31,1
100008,1253167,2025-12-31,1
100010,1253167,2025-12-31,1


In [20]:
# Portfolio sample: tblSales

# Columns to display
display_columns = [
    "product_id",
    "snapshot_date",
    "qty_on_hand"
]

# Styled table
styled = style_report_table(
    tblInventorySnapshot[display_columns].head(),
    left_align_cols=["product_id"]
)

# Hide row index
styled = styled.hide(axis="index")

# Verticle alighnment: bottom
styled = styled.set_table_styles(
    [
        {
            "selector": "th.col_heading",
            "props": [
                ("white-space", "pre-line"),
                ("vertical-align", "bottom")
            ]
        }
    ],
    overwrite=False
)

# Business-friendly column labels
rename_map = {
    "product_id": "Prod ID",
    "snapshot_date": "Snapshot\nDate",
    "qty_on_hand": "Qty\nOn Hand"
}

# Display title
display(HTML("<h3 style='margin-bottom: 6px;'>Inventory Snapshot</h3>"))

# Display renamed/styled table
display(
    styled.relabel_index(
        [rename_map.get(col, col) for col in tblInventorySnapshot[display_columns].columns],
        axis=1
    )
)

Prod ID,Snapshot Date,Qty On Hand
1088638,2025-12-31,30
1253167,2025-12-31,12
1331817,2025-12-31,30
1711217,2025-12-31,26
2012714,2025-12-31,14
